# <center> Exotic Option Pricing via Least-Squares Monte Carlo 

## <center> Nayan Adani

## Import and parameters setting

In [ ]:
import numpy as np
from scipy.stats import norm
import pandas as pd
import matplotlib.pyplot as plt

N = 100000
S0 = 100
K = 98
r = 0
q = 0.02
sigma = 0.23

times = [0.25, 0.5, 0.75, 1.0]

## Simulate path

#### This section simulates stock paths under the Black–Scholes dynamics at the Bermudan exercise dates.

In [3]:
def simulate_paths_bs(S0, r, q, sigma, times, N):
    dt = np.diff([0] + times)
    paths = np.zeros((N, len(times)))
    S = np.full(N, S0)

    for i, delta in enumerate(dt):
        Z = np.random.randn(N)
        S = S * np.exp((r - q - 0.5 * sigma**2) * delta + sigma * np.sqrt(delta) * Z)
        paths[:, i] = S

    return paths

In [4]:
paths_S = simulate_paths_bs(S0, r, q, sigma, times, N)

In [5]:
paths_A = np.zeros_like(paths_S)

paths_A[:,0] = paths_S[:,0]
paths_A[:,1] = (paths_S[:,0] + paths_S[:,1]) / 2
paths_A[:,2] = (paths_S[:,0] + paths_S[:,1] + paths_S[:,2]) / 3
paths_A[:,3] = (paths_S[:,0] + paths_S[:,1] + paths_S[:,2] + paths_S[:,3]) / 4

## LSMC

In [ ]:
K = 98
payoffs = np.maximum(paths_A - K, 0.0)   # shape (N, 4)

cashflows = payoffs[:, -1].copy()   # last exercise date (maturity)

In [ ]:
def lsmc_price(paths_S, paths_A, K, r=0.0):
    
    '''This function implements the Longstaff–Schwartz backward induction: at each exercise date, 
    continuation values are estimated by polynomial regression on in-the-money paths.'''
    
    N, M = paths_S.shape   # N paths, M exercise dates
    payoffs = np.maximum(paths_A - K, 0.0)

    # Initialize cashflows with terminal payoffs (at maturity)
    cashflows = payoffs[:, -1].copy()

    # Backward induction: process dates M-2, ..., 0
    for j in range(M - 2, -1, -1):

        # Identify in-the-money (ITM) paths at date j
        itm = payoffs[:, j] > 0.0
        if not np.any(itm):
            continue

        # State variables for ITM paths
        S = paths_S[itm, j]
        A = paths_A[itm, j]

        # Construct regression features: [1, S, S^2, S^3, A, A^2, A^3]
        X = np.column_stack([
            np.ones_like(S),
            S, S**2, S**3,
            A, A**2, A**3
        ])

        # Future discounted cashflows (discount = 1 here because r=0)
        Y = cashflows[itm]

        # Ordinary least squares regression
        beta, *_ = np.linalg.lstsq(X, Y, rcond=None)

        # Estimated continuation value
        continuation = X @ beta

        # Exercise decision: compare immediate payoff vs continuation
        exercise_now = payoffs[itm, j] > continuation

        # Update cashflows: replace continuation by immediate exercise payoff
        cashflows[itm] = np.where(
            exercise_now,
            payoffs[itm, j],
            cashflows[itm]
        )
            # Monte Carlo estimate
    price = np.mean(cashflows)

    # Standard error of the Monte Carlo estimator
    std_dev = np.std(cashflows, ddof=1)      # sample standard deviation
    std_error = std_dev / np.sqrt(N)

    # 95% confidence interval (normal approximation)
    z = 1.96
    ci_low = price - z * std_error
    ci_high = price + z * std_error

    return price, std_error, ci_low, ci_high

In [9]:
price, std_error, ci_low, ci_high = lsmc_price(paths_S, paths_A, K)

print("Price:", price)
print("Std error:", std_error)
print("95% CI:", ci_low, ci_high)

Price: 6.903023464769496
Std error: 0.03222387989368268
95% CI: 6.839864660177878 6.966182269361114


## Using the local volatility model 

In [26]:
import numpy as np

# Product and market parameters 
S0 = 100.0
K = 98.0
r = 0.0          
q = 0.02         

T = 1.0
exercise_times = np.array([0.25, 0.5, 0.75, 1.0])

N_PATHS = 100_000    # number of Monte Carlo paths
DT = 1.0 / 52.0      # weekly time step for Euler scheme

In [27]:
import Assignment3_groupH as AH  

tau = AH.tau               
strikes = AH.strikes             
has_iv = AH.has_iv                 
v_levels_per_expiry = AH.v_levels_per_expiry
build_theta_from_levels = AH.build_theta_from_levels

nT = len(tau)
I = len(strikes)

print("Loaded AH calibration:")
print("  maturities tau =", tau)
print("  #strike grid points I =", I)
print("  #expiry slabs nT =", nT)


Loaded AH calibration:
  maturities tau = [0.025 0.101 0.197 0.274 0.523 0.772 1.769]
  #strike grid points I = 161
  #expiry slabs nT = 7


In [28]:
quoted_idx_list = [np.where(has_iv[:, j])[0] for j in range(nT)]

theta_grid_per_expiry = np.zeros((nT, I))
for j in range(nT):
    quoted_idx = quoted_idx_list[j]
    theta_grid_per_expiry[j, :] = build_theta_from_levels(
        v_levels_per_expiry[j], quoted_idx, I
    )

K_min, K_max = strikes[0], strikes[-1]

def sigma_local(S, t):
    """Local volatility σ_L(S,t) derived from the AH calibrated θ(K,T).

    Parameters
    ----------
    S : array-like or scalar
        Current stock level(s).
    t : float
        Current time, in years.

    Returns
    -------
    sigma : ndarray or float
        Local volatility evaluated at (S,t).
    """

    S_arr = np.asarray(S, dtype=float)

    if t <= 0.0:
        slab_idx = 0
    else:
        slab_idx = np.searchsorted(tau, t)
        slab_idx = min(slab_idx, nT - 1)

    theta = theta_grid_per_expiry[slab_idx, :]

    S_clip = np.clip(S_arr, K_min, K_max)

    theta_S = np.interp(S_clip, strikes, theta)

    sigma = theta_S / np.maximum(S_clip, 1e-8)
    sigma = np.maximum(sigma, 1e-4)

    return sigma


In [29]:
def simulate_paths_localvol(
    S0, r, q, exercise_times, N_paths, dt, sigma_local_func
):
    T = float(exercise_times[-1])
    n_steps = int(np.round(T / dt))
    t_grid = np.linspace(0.0, T, n_steps + 1)

    ex_indices = [np.argmin(np.abs(t_grid - t_ex)) for t_ex in exercise_times]

    paths_S = np.zeros((N_paths, len(exercise_times)))
    S = np.full(N_paths, S0, dtype=float)

    ex_counter = 0
    for step in range(n_steps):
        t = t_grid[step]

        # Local volatility at current state
        sigma_t = sigma_local_func(S, t)

        # Brownian increment
        Z = np.random.randn(N_paths)

        # Euler step under Q
        S = S + (r - q) * S * dt + sigma_t * S * np.sqrt(dt) * Z

        S = np.maximum(S, 1e-8)

        if step + 1 in ex_indices:
            paths_S[:, ex_counter] = S
            ex_counter += 1

    return paths_S

np.random.seed(123)  
paths_S_lv = simulate_paths_localvol(
    S0, r, q, exercise_times, N_PATHS, DT, sigma_local
)

paths_S_lv.shape


(100000, 4)

In [30]:
paths_A_lv = np.zeros_like(paths_S_lv)

paths_A_lv[:, 0] = paths_S_lv[:, 0]
paths_A_lv[:, 1] = (paths_S_lv[:, 0] + paths_S_lv[:, 1]) / 2.0
paths_A_lv[:, 2] = (paths_S_lv[:, 0] + paths_S_lv[:, 1] + paths_S_lv[:, 2]) / 3.0
paths_A_lv[:, 3] = (
    paths_S_lv[:, 0] + paths_S_lv[:, 1] + paths_S_lv[:, 2] + paths_S_lv[:, 3]
) / 4.0

paths_A_lv.shape

(100000, 4)

In [31]:
def lsmc_price(paths_S, paths_A, K, r=0.0, exercise_times=None):
    N, M = paths_S.shape
    payoffs = np.maximum(paths_A - K, 0.0)   # immediate exercise payoff

    # terminal payoffs
    cashflows = payoffs[:, -1].copy()

    if exercise_times is not None and r != 0.0:
        exercise_times = np.asarray(exercise_times)
        dt_ex = np.diff(np.concatenate([[0.0], exercise_times]))
        disc_step = np.exp(-r * dt_ex)   
    else:
        disc_step = np.ones(M)

    # Backward induction over exercise dates j = M-2, ..., 0
    for j in range(M - 2, -1, -1):
        # Discount one step back
        cashflows *= disc_step[j + 1]

        itm = payoffs[:, j] > 0.0  # in-the-money indicator
        if not np.any(itm):
            continue

        S_j = paths_S[itm, j]
        A_j = paths_A[itm, j]

        # Regression design matrix
        X = np.column_stack([
            np.ones_like(S_j),
            S_j, S_j**2, S_j**3,
            A_j, A_j**2, A_j**3
        ])

        Y = cashflows[itm]

        # Ordinary least squares regression
        beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
        continuation = X @ beta

        # Early exercise decision
        exercise_now = payoffs[itm, j] > continuation

        cashflows[itm] = np.where(exercise_now, payoffs[itm, j], cashflows[itm])

    if exercise_times is not None and r != 0.0:
        cashflows *= disc_step[0]

    # Monte Carlo estimator: mean + standard error and 95% CI
    price = np.mean(cashflows)
    std_dev = np.std(cashflows, ddof=1)
    std_error = std_dev / np.sqrt(N)

    z = 1.96
    ci_low = price - z * std_error
    ci_high = price + z * std_error

    return price, std_error, ci_low, ci_high


In [32]:
price_lv, se_lv, ci_low_lv, ci_high_lv = lsmc_price(
    paths_S_lv, paths_A_lv, K, r=r, exercise_times=exercise_times
)

print(f"Local-vol LSMC price: {price_lv:.6f}")
print(f"Standard error:       {se_lv:.6f}")
print(f"95% CI:               [{ci_low_lv:.6f}, {ci_high_lv:.6f}]")


Local-vol LSMC price: 8.073079
Standard error:       0.027343
95% CI:               [8.019486, 8.126672]
